# Top by-magnitude features, laid out across all 12 categories

For each of the 28 distinct features that show up in any category's top-10-by-magnitude
(see `sae_top10_heatmaps.ipynb` / `heatmaps/top10_by_category_magnitude/top10_summary.csv`),
render one figure per race (6 races) showing that feature's activation heatmap across
9 sampled seeds for men (left) and 9 for women (right), so you can compare how the same
feature fires across the full population instead of only within a single category.

Outputs: `baseline/heatmaps/top_features_by_race/feature<idx>/<race>.png`
(28 features x 6 races = 168 images).

In [1]:
import os
os.chdir('/n/fs/goose/ReNO')

import sys
sys.path.append('/n/fs/goose/sdxl-unbox')

import re
import csv
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image
from pytorch_lightning import seed_everything

from diffusers import AutoencoderKL, EulerAncestralDiscreteScheduler
from SDLens import HookedStableDiffusionXLPipeline
from SAE import SparseAutoencoder

def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.2.2+cu121)
    Python  3.10.14 (you have 3.10.20)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


In [2]:
# Config
CACHE_DIR = 'hf_cache'
N_INFERENCE_STEPS = 1
DTYPE = torch.float16
DEVICE = torch.device('cuda')

CODE = 'down.2.1'
CODE_TO_BLOCK = {
    "down.2.1": "unet.down_blocks.2.attentions.1",
    "mid.0": "unet.mid_block.attentions.0",
    "up.0.0": "unet.up_blocks.0.attentions.0",
    "up.0.1": "unet.up_blocks.0.attentions.1",
}
CHECKPOINT_DIR = '/n/fs/goose/sdxl-unbox/checkpoints'
OUT_DIR = '/n/fs/goose/baseline/heatmaps/top_features_by_race'
os.makedirs(OUT_DIR, exist_ok=True)

TOP10_CSV = '/n/fs/goose/baseline/heatmaps/top10_by_category_magnitude/top10_summary.csv'

RACES = ["black", "white", "east asian", "middle eastern", "indian", "hispanic"]
GENDERS = ["man", "woman"]
CATEGORIES = [(race, gender) for gender in GENDERS for race in RACES]  # 12 categories

PROMPT_TEMPLATE = (
    "a portrait of a {race} {gender} with a neutral expression, "
    "without any facial coverings, in a neutral white colored t-shirt against a neutral white background"
)

N_SAMPLE_SEEDS = 9   # seeds sampled per category, shown as a 3x3 grid
RNG_SEED = 0         # for picking the 9 seeds per category, reproducibly

def cat_slug(race, gender):
    return slugify(f"{race}_{gender}")

def feature_slug(feat_idx):
    return f"feature{feat_idx}"

# --- features to visualize: every distinct feature_idx across all categories' top-10-by-magnitude ---
with open(TOP10_CSV) as f:
    FEATURES = sorted({int(row["feature_idx"]) for row in csv.DictReader(f)})
print(f"{len(FEATURES)} distinct features:", FEATURES)

28 distinct features: [138, 452, 472, 527, 546, 669, 839, 1584, 1653, 1664, 1678, 2055, 2182, 2611, 2670, 2707, 2808, 2893, 2940, 3279, 3701, 3764, 3894, 4511, 4544, 4858, 4957, 5046]


In [3]:
hooked_vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", torch_dtype=DTYPE, cache_dir=CACHE_DIR,
)
hooked_pipe = HookedStableDiffusionXLPipeline.from_pretrained(
    "stabilityai/sdxl-turbo", vae=hooked_vae, torch_dtype=DTYPE, variant="fp16",
    use_safetensors=True, cache_dir=CACHE_DIR,
)
hooked_pipe.pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    hooked_pipe.pipe.scheduler.config, timestep_spacing="trailing",
)
hooked_pipe.pipe = hooked_pipe.pipe.to(DEVICE, DTYPE)

sae = SparseAutoencoder.load_from_disk(
    os.path.join(CHECKPOINT_DIR, f"{CODE_TO_BLOCK[CODE]}_k10_hidden5120_auxk256_bs4096_lr0.0001", "final")
).to(DEVICE)
n_dirs = sae.n_dirs
print("n_dirs:", n_dirs)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:  14%|█▍        | 1/7 [00:19<01:59, 19.85s/it]

Loading pipeline components...:  29%|██▊       | 2/7 [00:26<01:01, 12.23s/it]

Loading pipeline components...:  43%|████▎     | 3/7 [00:27<00:27,  6.84s/it]

Loading pipeline components...:  86%|████████▌ | 6/7 [00:28<00:02,  2.60s/it]

Loading pipeline components...: 100%|██████████| 7/7 [00:31<00:00,  2.64s/it]

Loading pipeline components...: 100%|██████████| 7/7 [00:31<00:00,  4.46s/it]

n_dirs: 5120


In [4]:
def get_diff_and_image(prompt, seed):
    """Replays the deterministic (seed_everything, latents, generator) recipe used
    throughout baseline/, returning the generated image and the block's residual
    delta (output - input), shaped [h, w, d_model]."""
    seed_everything(seed)
    generator = torch.Generator("cuda").manual_seed(seed)
    latents = torch.randn((1, 4, 64, 64), device=DEVICE, dtype=DTYPE)

    with torch.no_grad():
        images, cache = hooked_pipe.run_with_cache(
            prompt,
            latents=latents,
            generator=generator,
            num_inference_steps=N_INFERENCE_STEPS,
            guidance_scale=0.0,
            positions_to_cache=[CODE_TO_BLOCK[CODE]],
            save_input=True,
            save_output=True,
        )
    image = images.images[0]

    block = CODE_TO_BLOCK[CODE]
    diff = cache["output"][block] - cache["input"][block]
    if diff.shape[0] == 2:  # classifier-free guidance batch: keep the conditional half
        diff = diff[1].unsqueeze(0)
    diff_last = diff[:, -1].permute(0, 2, 3, 1).squeeze(0)  # [h, w, d_model]
    return image, diff_last

In [5]:
# --- Step A: for each of the 12 categories, sample N_SAMPLE_SEEDS seeds and cache
# per-seed (image, feats[h, w, n_dirs]) so every feature's grid reuses the same samples ---
rng = np.random.default_rng(RNG_SEED)
category_samples = {}  # cat -> list of (image, feats[h, w, n_dirs], seed)

for cat in CATEGORIES:
    race, gender = cat
    prompt = PROMPT_TEMPLATE.format(race=race, gender=gender)
    chosen_seeds = rng.choice(1000, size=N_SAMPLE_SEEDS, replace=False).tolist()

    samples = []
    for seed in chosen_seeds:
        image, diff_last = get_diff_and_image(prompt, seed)
        h, w, d_model = diff_last.shape
        with torch.no_grad():
            feats = sae.encode(diff_last.reshape(-1, d_model).float().to(DEVICE))
        feats = feats.reshape(h, w, n_dirs).cpu().numpy()
        samples.append((image, feats, seed))
    category_samples[cat] = samples
    print(f"done sampling category={cat}")

[rank: 0] Seed set to 843


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:10<00:00, 10.37s/it]

100%|██████████| 1/1 [00:10<00:00, 10.37s/it]

[rank: 0] Seed set to 632


...

100%|██████████| 1/1 [00:00<00:00, 14.66it/s]


[rank: 0] Seed set to 823


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 14.69it/s]

done sampling category=('hispanic', 'woman')


In [6]:
def make_heatmap_overlay(image: Image.Image, feature_map: np.ndarray, upscale: int) -> Image.Image:
    """Same jet/alpha-composite recipe as sdxl-unbox/app.py:plot_image_heatmap."""
    heatmap = np.kron(feature_map, np.ones((upscale, upscale)))
    image = image.convert("RGBA")

    jet = plt.cm.jet
    cmap = jet(np.arange(jet.N))
    cmap[:1, -1] = 0
    cmap[1:, -1] = 0.6
    cmap = ListedColormap(cmap)

    denom = np.max(heatmap) - np.min(heatmap)
    heatmap = (heatmap - np.min(heatmap)) / denom if denom > 0 else np.zeros_like(heatmap)
    heatmap_rgba = cmap(heatmap)
    heatmap_image = Image.fromarray((heatmap_rgba * 255).astype(np.uint8)).resize(image.size)

    return Image.alpha_composite(image, heatmap_image)

In [7]:
# --- Step B: for each feature x race, render men (3x3) + women (3x3) side by side ---
for feat_idx in FEATURES:
    feat_dir = os.path.join(OUT_DIR, feature_slug(feat_idx))
    os.makedirs(feat_dir, exist_ok=True)

    for race in RACES:
        fig, axes = plt.subplots(3, 6, figsize=(18, 9))

        for gender_i, gender in enumerate(GENDERS):
            samples = category_samples[(race, gender)]  # 9 x (image, feats, seed)
            col_offset = gender_i * 3
            for s_i, (image, feats, seed) in enumerate(samples):
                r, c = divmod(s_i, 3)
                ax = axes[r, col_offset + c]
                feature_map = feats[:, :, feat_idx]
                upscale = image.size[0] // feature_map.shape[1]
                overlay = make_heatmap_overlay(image, feature_map, upscale)
                ax.imshow(overlay)
                ax.set_title(f"seed={seed}  max={feature_map.max():.2f}", fontsize=7)
                ax.set_aspect("equal")
                ax.axis("off")

        axes[0, 1].annotate("MEN", xy=(0.5, 1.15), xycoords="axes fraction",
                             ha="center", fontsize=13, fontweight="bold")
        axes[0, 4].annotate("WOMEN", xy=(0.5, 1.15), xycoords="axes fraction",
                             ha="center", fontsize=13, fontweight="bold")
        fig.suptitle(f"feature {feat_idx} ({CODE}) -- {race}", y=1.04, fontsize=14)
        fig.tight_layout()

        out_path = os.path.join(feat_dir, f"{slugify(race)}.png")
        fig.savefig(out_path, dpi=110, bbox_inches="tight")
        plt.close(fig)

    print(f"done feature={feat_idx}")

done feature=138


done feature=452


done feature=472


done feature=527


done feature=546


...

done feature=4511


done feature=4544


done feature=4858


done feature=4957


done feature=5046
